# Building a RAG application from scratch

Here is a high-level overview of the system we want to build:

<img src='images/system1.png' width="1200">

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
PINECONE_API_KEY = os.environ["PINECONE_API_KEY"]

# This is the YouTube video/podcast we're going to use.
YOUTUBE_VIDEO = "https://www.youtube.com/watch?v=cdiD-9MMpb0"

## Setting up the model
Let's define the LLM model that we'll use as part of the workflow. `ChatOpenAI` picks up `OPENAI_API_KEY` from the environment automatically, so we don't need to pass it (or any literal key) in code.

In [ ]:
from langchain_openai.chat_models import ChatOpenAI

model = ChatOpenAI(model="gpt-3.5-turbo")

We can test the model by asking a simple question.

In [ ]:
model.invoke("What MLB team won the World Series during the COVID-19 pandemic?")

The result from the model is an `AIMessage` instance containing the answer. We can extract this answer by chaining the model with an [output parser](https://python.langchain.com/docs/modules/model_io/output_parsers/).

Here is what chaining the model with an output parser looks like:

<img src='images/chain1.png' width="1200">

For this example, we'll use a simple `StrOutputParser` to extract the answer as a string.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

chain = model | parser
chain.invoke("What MLB team won the World Series during the COVID-19 pandemic?")

## Introducing prompt templates

We want to provide the model with some context and the question. [Prompt templates](https://python.langchain.com/docs/modules/model_io/prompts/quick_start) are a simple way to define and reuse prompts.

In [ ]:
from langchain.prompts import ChatPromptTemplate

template = """
Answer the question based on the context below. If you can't
answer the question, reply "I don't know".

Context: {context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)
prompt.format(context="Mary's sister is Susana", question="Who is Mary's sister?")

We can now chain the prompt with the model and the output parser.

<img src='images/chain2.png' width="1200">

In [ ]:
chain = prompt | model | parser
chain.invoke({
    "context": "Mary's sister is Susana",
    "question": "Who is Mary's sister?"
})

## Combining chains

We can combine different chains to create more complex workflows. For example, let's create a second chain that translates the answer from the first chain into a different language.

Let's start by creating a new prompt template for the translation chain:

In [ ]:
translation_prompt = ChatPromptTemplate.from_template(
    "Translate {answer} to {language}"
)

We can now create a new translation chain that combines the result from the first chain with the translation prompt.

Here is what the new workflow looks like:

<img src='images/chain3.png' width="1200">

In [ ]:
from operator import itemgetter

translation_chain = (
    {"answer": chain, "language": itemgetter("language")} | translation_prompt | model | parser
)

translation_chain.invoke(
    {
        "context": "Mary's sister is Susana. She doesn't have any more siblings.",
        "question": "How many sisters does Mary have?",
        "language": "Spanish",
    }
)

## Transcribing the YouTube video/podcast

The context we want to send the model comes from a YouTube video or podcast. Instead of base [OpenAI Whisper](https://openai.com/research/whisper), we use **WhisperX** for this step, because WhisperX adds forced alignment and **speaker diarization** on top of Whisper's transcription. That matters specifically for podcasts, which often have crosstalk and multiple speakers — diarization gives much cleaner segment boundaries to clean/chunk in the next step than plain Whisper output does.

This logic lives in `transcript_preprocessing.py` (`transcribe_with_whisperx`) rather than inline, so it can be unit-tested independently of downloading a video.

In [ ]:
import os
import tempfile

from pytube import YouTube

from transcript_preprocessing import transcribe_with_whisperx

# Let's do this only if we haven't created the transcription file yet.
if not os.path.exists("transcription_segments.json"):
    youtube = YouTube(YOUTUBE_VIDEO)
    audio = youtube.streams.filter(only_audio=True).first()

    with tempfile.TemporaryDirectory() as tmpdir:
        file = audio.download(output_path=tmpdir)
        segments = transcribe_with_whisperx(file)

        import json
        with open("transcription_segments.json", "w") as f:
            json.dump([s.__dict__ for s in segments], f)

Let's read the diarized segments back and display the first couple to confirm everything worked as expected — note each segment now carries a `speaker` label, which base Whisper didn't give us.

In [ ]:
import json

from transcript_preprocessing import TranscriptSegment

with open("transcription_segments.json") as f:
    raw_segments = [TranscriptSegment(**s) for s in json.load(f)]

raw_segments[:2]

## Cleaning and chunking the transcript

If we try to invoke the chain using the whole transcript as context, the model will return an error because the context is too long — LLMs support limited context sizes, and a full podcast transcript is too long to hand over directly.

The original version of this notebook split the raw transcript with `RecursiveCharacterTextSplitter`, purely on character count. That's replaced here with custom preprocessing (`transcript_preprocessing.py`) that:

1. **Cleans** the transcript — strips filler words ("um", "like", "basically", ...) and low-signal segments, and drops immediate repeated lines (a common ASR artifact).
2. **Chunks** the cleaned transcript with a **sliding window and ~20% overlap**, rather than splitting on a fixed character count with no regard for word boundaries. The overlap keeps a sentence or idea from being cut in half at a chunk boundary, which was found (empirically, by varying chunk size / overlap / Pinecone's `top_k`) to meaningfully improve retrieval quality.

Everything here is standard Python — no separate ML model is trained for cleaning; WhisperX did the transcription/diarization work, and this step is just text processing on top of it.

In [ ]:
from transcript_preprocessing import clean_transcript, chunk_transcript, chunks_to_documents

cleaned_segments = clean_transcript(raw_segments, min_words=3)
chunks = chunk_transcript(cleaned_segments, chunk_size_words=200, overlap_ratio=0.20)
documents = chunks_to_documents(chunks, source="transcription_segments.json")

print(f"{len(raw_segments)} raw segments -> {len(cleaned_segments)} cleaned segments -> {len(documents)} chunks")
documents[:2]

## Finding the relevant chunks

Given a particular question, we need to find the relevant chunks from the transcript to send to the model. Here is where the idea of **embeddings** comes into play.

An embedding is a mathematical representation of the semantic meaning of a word, sentence, or document. It's a projection of a concept in a high-dimensional space. Embeddings have a simple characteristic: the projection of related concepts will be close to each other, while concepts with different meanings will lie far away.

**Note on vectorization**: OpenAI's embedding model is the only thing used to turn text into vectors in this project — Pinecone (below) only stores and searches those vectors, it doesn't generate embeddings itself. The same embedding model is used for both transcript chunks and incoming questions so they land in the same vector space.

To provide the most relevant chunks, we use the embeddings of the question and the chunks of the transcript to compute similarity between them, then select the chunks with the highest similarity to the question and use them as context for the model:

<img src='images/system3.png' width="1200">

Let's generate embeddings for an arbitrary query:

In [ ]:
from langchain_openai.embeddings import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()
embedded_query = embeddings.embed_query("Who is Mary's sister?")

print(f"Embedding length: {len(embedded_query)}")
print(embedded_query[:10])

To illustrate how embeddings work, let's first generate the embeddings for two different sentences:

In [ ]:
sentence1 = embeddings.embed_query("Mary's sister is Susana")
sentence2 = embeddings.embed_query("Pedro's mother is a teacher")

We can now compute the similarity between the query and each of the two sentences. The closer the embeddings are, the more similar the sentences will be.

We can use [Cosine Similarity](https://en.wikipedia.org/wiki/Cosine_similarity) to calculate the similarity between the query and each of the sentences:

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

query_sentence1_similarity = cosine_similarity([embedded_query], [sentence1])[0][0]
query_sentence2_similarity = cosine_similarity([embedded_query], [sentence2])[0][0]

query_sentence1_similarity, query_sentence2_similarity

## Setting up a Vector Store

We need an efficient way to store document chunks, their embeddings, and perform similarity searches at scale. To do this, we'll use a **vector store**.

A vector store is a database of embeddings that specializes in fast similarity searches.

<img src='images/system4.png' width="1200">

To understand how a vector store works, let's create one in memory and add a few embeddings to it:

In [ ]:
from langchain_community.vectorstores import DocArrayInMemorySearch

vectorstore1 = DocArrayInMemorySearch.from_texts(
    [
        "Mary's sister is Susana",
        "John and Tommy are brothers",
        "Patricia likes white cars",
        "Pedro's mother is a teacher",
        "Lucia drives an Audi",
        "Mary has two siblings",
    ],
    embedding=embeddings,
)

We can now query the vector store to find the most similar embeddings to a given query:

In [ ]:
vectorstore1.similarity_search_with_score(query="Who is Mary's sister?", k=3)

## Connecting the vector store to the chain

We can use the vector store to find the most relevant chunks from the transcript to send to the model. Here is how we can connect the vector store to the chain:

<img src='images/chain4.png' width="1200">

We need to configure a [Retriever](https://python.langchain.com/docs/modules/data_connection/retrievers/). The retriever will run a similarity search in the vector store and return the most similar documents back to the next step in the chain.

We can get a retriever directly from the vector store we created before:

In [ ]:
retriever1 = vectorstore1.as_retriever()
retriever1.invoke("Who is Mary's sister?")

Our prompt expects two parameters, "context" and "question." We can use the retriever to find the chunks we'll use as the context to answer the question.

We can create a map with the two inputs by using the [`RunnableParallel`](https://python.langchain.com/docs/expression_language/how_to/map) and [`RunnablePassthrough`](https://python.langchain.com/docs/expression_language/how_to/passthrough) classes.

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

setup = RunnableParallel(context=retriever1, question=RunnablePassthrough())
setup.invoke("What color is Patricia's car?")

Let's now add the setup map to the chain and run it:

In [ ]:
chain = setup | prompt | model | parser
chain.invoke("What color is Patricia's car?")

Let's invoke the chain using another example:

In [ ]:
chain.invoke("What car does Lucia drive?")

## Loading the transcript into the vector store

We initialized the vector store with a few random strings. Let's create a new vector store using the cleaned, sliding-window chunks from the podcast transcript we prepared earlier.

In [ ]:
vectorstore2 = DocArrayInMemorySearch.from_documents(documents, embeddings)

Let's set up a new chain using the correct vector store:

In [ ]:
chain = (
    {"context": vectorstore2.as_retriever(), "question": RunnablePassthrough()}
    | prompt
    | model
    | parser
)
chain.invoke("What is this podcast about?")

## Setting up Pinecone

So far we've used an in-memory vector store. In practice, we need a vector store that can handle large amounts of data and perform similarity searches at scale. For this example, we'll use [Pinecone](https://www.pinecone.io/).

The first step is to create a Pinecone account, set up an index, get an API key, and set it as an environment variable `PINECONE_API_KEY` in `.env` (already loaded at the top of this notebook — no key is hardcoded here).

Then, we load the cleaned/chunked transcript documents into Pinecone:

In [ ]:
from langchain_pinecone import PineconeVectorStore

index_name = "podcast-rag-index"

pinecone = PineconeVectorStore.from_documents(
    documents, embeddings, index_name=index_name
)

Let's now run a similarity search on Pinecone to make sure everything works:

In [ ]:
pinecone.similarity_search("What is this podcast about?")[:3]

Let's set up the final chain using Pinecone as the vector store. `top_k` (passed via `search_kwargs`) and the chunk size / overlap set above (`chunk_size_words=200`, `overlap_ratio=0.20`) are the three knobs worth tuning for retrieval quality — the embedding model itself stays fixed.

In [ ]:
chain = (
    {
        "context": pinecone.as_retriever(search_kwargs={"k": 5}),
        "question": RunnablePassthrough(),
    }
    | prompt
    | model
    | parser
)

chain.invoke("What is this podcast about?")

## Design notes

- **Vectorization**: OpenAI's embedding model is the only vectorization method used here; Pinecone stores and searches those vectors via cosine similarity. Retrieval quality was tuned by varying chunk size, chunk overlap, and Pinecone's `top_k` (`search_kwargs={"k": ...}`), not by swapping embedding models.
- **Transcript cleaning**: WhisperX transcribes and diarizes; a custom cleaning pass then strips filler words, drops near-empty/repeated segments, and only then do we chunk. Skipping this and chunking raw ASR output tends to produce noisier vectors and worse retrieval ranking.
- **Custom vs. off-the-shelf**: WhisperX is used as-is. The cleaning + sliding-window chunking logic in `transcript_preprocessing.py` is custom.